# A3.3.4: SQL aggregate functions

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* Core for SL and HL. Command term: **construct**. Builds on A3.3.2 and A3.3.3.

*Every query so far returned rows. An aggregate function reads many rows and returns one calculated value.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-4

In [ ]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

In [ ]:
# Install the SQL add-on into THIS kernel if it is missing (one time only). (Given.)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sql") is None:
    print("Installing the SQL add-on (one time, about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"])
    importlib.invalidate_caches()

import sqlite3
db = sqlite3.connect(":memory:")
db.executescript('''
CREATE TABLE Mechanic (MechanicID TEXT PRIMARY KEY, MechanicName TEXT, Station TEXT);
CREATE TABLE Repair (RepairID TEXT PRIMARY KEY, BikeType TEXT, MechanicID TEXT, PartsCost REAL, LabourHours REAL,
                     FOREIGN KEY (MechanicID) REFERENCES Mechanic (MechanicID));
INSERT INTO Mechanic VALUES ('M1', 'Nia Halvorsen', 'Front bench'), ('M2', 'Tomas Bright', 'Wheel bay'), ('M3', 'Ola Fenn', 'Front bench');
INSERT INTO Repair VALUES ('R1', 'Road', 'M1', 45, 1.5), ('R2', 'Mountain', 'M2', 120, 3),
                          ('R3', 'Road', 'M2', 30, 1), ('R4', 'Electric', 'M1', 260, 4.5),
                          ('R5', 'Mountain', 'M3', 85, 2), ('R6', 'Electric', 'M3', 310, 5),
                          ('R7', 'Road', 'M1', 55, 2), ('R8', 'Mountain', 'M1', 95, 2.5);
''')

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

## 1. The five functions

An **aggregate function** reaches across all the matching records and hands back a **single** number. The five on the syllabus are `COUNT`, `SUM`, `AVERAGE`, `MIN` and `MAX`. Four need a numeric field (you cannot total text). `COUNT` counts records, so it works on any field.

> **AVERAGE or AVG?** The syllabus says AVERAGE, but real databases implement it as `AVG(...)`. Write `AVG` in the query and use the word "average" in explanations. Exact syntax varies between database systems, and a marker reads for the right function.

A bike shop keeps repairs. Here are the tables.

In [ ]:
%%sql
SELECT * FROM Repair;

In [ ]:
%%sql
SELECT * FROM Mechanic;

In [ ]:
%%sql
SELECT COUNT(RepairID) AS NumRepairs,
       SUM(PartsCost)  AS TotalParts,
       AVG(PartsCost)  AS AveragePartsCost,
       MIN(PartsCost)  AS Cheapest,
       MAX(PartsCost)  AS Dearest
FROM Repair;

One row for the whole table. "Total parts cost: 1000" tells the owner very little, though.

## 2. How GROUP BY changes the answer

Without `GROUP BY` the whole table is **one pile** and you get one row. With `GROUP BY` the rows are first sorted into piles by a column, and the function runs separately on each pile. Three bike types gives three answers.

In [ ]:
%%sql
SELECT BikeType, COUNT(RepairID) AS NumRepairs, SUM(PartsCost) AS TotalParts
FROM Repair
GROUP BY BikeType;

"Electric bikes: 570.00 across only 2 repairs" tells the owner what to stock and what to charge. That is the reporting value of `GROUP BY`.

Aggregates join with the rest of `SELECT`: a `JOIN` first, then `GROUP BY`, then `HAVING` to filter the groups.

In [ ]:
%%sql
SELECT Mechanic.MechanicName, COUNT(*) AS Jobs, SUM(Repair.PartsCost) AS PartsUsed
FROM Repair
JOIN Mechanic ON Repair.MechanicID = Mechanic.MechanicID
GROUP BY Mechanic.MechanicName
HAVING COUNT(*) >= 3;

## 3. Try breaking things

Read each message. Each of these is a real exam error.

In [ ]:
def try_sql(text):
    try:
        print(db.execute(text).fetchall())
    except Exception as e:
        print("Error:", e)

print("1. An aggregate inside WHERE:")
try_sql("SELECT BikeType FROM Repair WHERE SUM(PartsCost) > 100")
print("2. AVERAGE instead of AVG:")
try_sql("SELECT AVERAGE(PartsCost) FROM Repair")
print("3. A loose column next to an aggregate, no GROUP BY:")
try_sql("SELECT BikeType, SUM(PartsCost) FROM Repair")

**Careful with number 3 and the loose column rule.** Every column in the `SELECT` list must be **either inside an aggregate or named in `GROUP BY`**. SQLite is lenient and quietly gives you one arbitrary value, but IB markers (and most other databases) do not. Look at what SQLite does here:

In [ ]:
%%sql
SELECT BikeType, MechanicID, SUM(PartsCost) FROM Repair GROUP BY BikeType;

The Mountain group had repairs by three different mechanics (M1, M2, M3). The `MechanicID` you see is just one of them. No correct value exists for that cell. Either drop the loose column or group by it too: `GROUP BY BikeType, MechanicID` gives one row per bike type per mechanic.

## 4. Your turn: construct the queries

**TODO 1.** Return the total parts cost of every repair (one value, call it `TotalParts`).

In [ ]:
%%sql q1 <<
-- TODO 1: SELECT SUM(PartsCost) AS TotalParts FROM Repair
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_rows("TODO 1", "q1", [(1000.0,)])

**TODO 2.** For each bike type, show `BikeType`, the number of repairs and the average parts cost.

In [ ]:
%%sql q2 <<
-- TODO 2: SELECT BikeType, COUNT(*), AVG(PartsCost) FROM Repair GROUP BY ...
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_rows("TODO 2", "q2", [("Road", 3, 43.33), ("Mountain", 3, 100.0), ("Electric", 2, 285.0)])

**TODO 3.** The cheapest and the most expensive repair in the table (`MIN` and `MAX` of `PartsCost`, in that order).

In [ ]:
%%sql q3 <<
-- TODO 3: SELECT MIN(PartsCost), MAX(PartsCost) FROM Repair
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_rows("TODO 3", "q3", [(30.0, 310.0)])

**TODO 4.** Join `Repair` and `Mechanic`. List each mechanic's name and their **highest single parts cost**, but only for mechanics who completed **more than two** repairs.

In [ ]:
%%sql q4 <<
-- TODO 4: JOIN, GROUP BY MechanicName, MAX(PartsCost), HAVING COUNT(*) > 2
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_rows("TODO 4", "q4", [("Nia Halvorsen", 260.0)])

**TODO 5.** Fix the broken query. It tries to show the mechanic with each bike-type total. Rewrite it so that every selected column is either aggregated or grouped (group by **both** `BikeType` and `MechanicID`).

Broken version: `SELECT BikeType, MechanicID, SUM(PartsCost) FROM Repair GROUP BY BikeType;`

In [ ]:
%%sql q5 <<
-- TODO 5: group by BikeType and MechanicID
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_rows("TODO 5", "q5", [("Electric", "M1", 260.0), ("Electric", "M3", 310.0), ("Mountain", "M1", 95.0), ("Mountain", "M2", 120.0), ("Mountain", "M3", 85.0), ("Road", "M1", 100.0), ("Road", "M2", 30.0)])

## 5. Check your understanding

Why does `SELECT BikeType, MechanicID, SUM(PartsCost) FROM Repair GROUP BY BikeType;` fail the rule? **A** SUM cannot be combined with GROUP BY, **B** MechanicID is neither aggregated nor grouped, so it has no single value per group, **C** GROUP BY must always come before SELECT, **D** a query can only ever select one column alongside an aggregate.

In [ ]:
choice = "?"   # type A, B, C or D
check_choice(choice, "B", "Every selected column must be aggregated or grouped. The Mountain group holds repairs by several mechanics, so one MechanicID for that output row is undefined.",
             "Look at what each column would have to contain. The Mountain group holds three repairs by three different mechanics.")

**Examiner note.** Constructed queries often select a column that is neither inside an aggregate nor in `GROUP BY`. It reads fine in English but asks for a value that does not exist. Check every column in the `SELECT` list against the rule before you write the query down.

## 6. Revision questions

Questions 1 to 3 are TODO 1, TODO 2 and TODO 4. Here is a written one.

**Q. Explain, using an example from the REPAIR table, why `WHERE SUM(PartsCost) > 100` is invalid and which clause should replace it.** [3 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)